# RF-DETR 74-Class Hidden45 Colab CUDA Training

## Goal

Train RF-DETR on the latest 74-class pill dataset using a Colab CUDA runtime instead of local Apple Silicon MPS.

Default path:

```text
Google Drive dataset tar.gz
  -> /content/rfdetr_colab/datasets/rfdetr_dataset_74_hidden45_canvas_balanced
  -> RF-DETR train/valid dry-run
  -> CUDA full train/resume
  -> mAP@0.75 checkpoint selection
```

The notebook keeps the existing RF-DETR parameters where possible: `epochs=12`, `seed=42`, cosine LR schedule, checkpoint every epoch, `eval_interval=1`, `eval_max_dets=100`, and best checkpoint selection by validation `mAP@0.75`.


## Colab Checklist

1. Set `Runtime > Change runtime type > GPU`.
2. Put the latest dataset archive in Drive, or use the default shared Drive file ID already embedded below.
   Mounted-path default: `/content/drive/MyDrive/ai12-level1-project/dataset_74_hidden45_latest_20260706/`
   Shared folder fallback: `https://drive.google.com/drive/folders/1eZp-iULcZhw30h-LExPvP-WRbnXtLzjv`
3. Run cells top to bottom once with `RUN_DRY_RUN=True` and `RUN_FULL_TRAIN=False`.
4. If the dry-run passes, set `RUN_FULL_TRAIN=True` and rerun the training cell.
5. Outputs and checkpoints are written under Drive so Colab disconnects can resume from `checkpoint_N.ckpt`.

Default dataset archive is the canvas-balanced package because it better matches the full test-image scale while keeping validation non-synthetic.


In [ ]:
# =========================
# Parameters - edit here
# =========================
from pathlib import Path

RUN_MOUNT_DRIVE = True
RUN_CLONE_OR_UPDATE_REPO = True
RUN_INSTALL_DEPS = True
RUN_EXTRACT_DATASET = True
FORCE_REEXTRACT_DATASET = False
RUN_DRY_RUN = True
RUN_FULL_TRAIN = False
RUN_FINALIZE_ONLY = False
RUN_SHOW_TENSORBOARD = False
RUN_TEST_INFERENCE = False
REQUIRE_DRIVE_FOR_CHECKPOINTS = True

TEAM_REPO_URL = 'https://github.com/kim-tae-yoon-0718/ai12-team01.git'
TEAM_REPO_BRANCH = 'model/rfdetr-basic74-45img-filled'

WORK_ROOT = Path('/content/rfdetr_colab')
REPO_DIR = WORK_ROOT / 'ai12-team01'
RFDETR_DIR = REPO_DIR / 'RF_DETR_split_ver'

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/ai12-level1-project')
DATASET_ARCHIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1eZp-iULcZhw30h-LExPvP-WRbnXtLzjv'
DATASET_ARCHIVE_FILE_ID = '1_DFR_eeEQVCnkF0NCYvIG6ESZFgo3rGa'
DATASET_ARCHIVE_FILE_URL = 'https://drive.google.com/file/d/1_DFR_eeEQVCnkF0NCYvIG6ESZFgo3rGa/view'
DATASET_ARCHIVE_DIR = DRIVE_PROJECT_DIR / 'dataset_74_hidden45_latest_20260706'
DATASET_ARCHIVE_NAME = 'dataset_74_hidden45_canvas_balanced_train_valid_20260706.tar.gz'
ARCHIVE_CACHE_DIR = WORK_ROOT / 'archives'
DATASET_EXTRACT_DIR = WORK_ROOT / 'datasets'
DATASET_DIR = DATASET_EXTRACT_DIR / 'rfdetr_dataset_74_hidden45_canvas_balanced'

CONFIG_NAME = 'config_74_hidden45_colab_cuda.yaml'
BASE_CONFIG_NAME = 'config_74_hidden45_canvas_balanced.yaml'
MODEL_VARIANT = 'nano'  # nano | small | medium | base | large

# Keep augmented runs in a separate output/checkpoint folder from the baseline.
BASE_MODEL_TAG = 'nano_74_hidden45_canvas_balanced_colab_cuda'
ENABLE_TRAIN_AUGMENTATION = True
AUGMENTATION_NAME = 'aug_scale150_rot90_v1'
MODEL_TAG = f'{BASE_MODEL_TAG}_{AUGMENTATION_NAME}' if ENABLE_TRAIN_AUGMENTATION else BASE_MODEL_TAG

# Original competition test images. The dataset archive may contain local symlinks,
# so the submission cells verify readability and fall back to this Drive folder.
TEST_IMAGES_FOLDER_URL = 'https://drive.google.com/drive/folders/1ZdGRPB3Xg4-1QKrKKKzzBOy7d2gorfuw'
TEST_IMAGES_FOLDER_ID = '1ZdGRPB3Xg4-1QKrKKKzzBOy7d2gorfuw'
TEST_DATA_DIR = WORK_ROOT / 'competition_test_images'
TEST_SCORE_THR = 0.05
TEST_MAX_DET_PER_IMAGE = 100
SUBMISSION_FILE_NAME = f'{MODEL_TAG}_submission.csv'

# Same effective batch as local MPS default: 1 x 16 = 16.
# CUDA default uses micro-batch 2 x grad_accum 8 = 16. If OOM, set 1 and 16.
EPOCHS = 12
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
NUM_WORKERS = 2       # raise to 4 only if Colab runtime and Drive I/O are stable
PIN_MEMORY = True
TRAIN_EPOCHS_OVERRIDE = None  # keep None for config-controlled epochs

# Online train-time augmentation. Flip is intentionally excluded because mirrored pill imprints are unrealistic.
# Start with this strong-rotation profile; lower AUG_ROTATE_LIMIT_DEGREES to 30/45 if validation becomes unstable.
AUG_SCALE_RANGE = [0.85, 1.50]
AUG_TRANSLATE_PERCENT = [-0.04, 0.04]
AUG_ROTATE_LIMIT_DEGREES = 90
AUG_AFFINE_P = 0.35
AUG_BRIGHTNESS_LIMIT = 0.08
AUG_CONTRAST_LIMIT = 0.08
AUG_COLOR_P = 0.25
ENABLE_LIGHT_AUGMENTATION = ENABLE_TRAIN_AUGMENTATION  # backward-compatible alias used below
ENABLE_MULTI_SCALE = False

OUTPUT_ROOT = DRIVE_PROJECT_DIR / 'rfdetr_outputs'
BACKUP_DIR = OUTPUT_ROOT / 'best'
SUBMISSION_DIR = OUTPUT_ROOT / MODEL_TAG / 'submissions'


In [ ]:
# Imports and small helpers
import json
import os
import random
import shutil
import subprocess
import sys
import tarfile
from collections import Counter
from pathlib import Path

import pandas as pd


def run(cmd, cwd=None, env=None):
    print('run:', ' '.join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True)


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


seed_everything(42)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print('WORK_ROOT:', WORK_ROOT)
print('Python:', sys.executable)


In [ ]:
# Mount Google Drive and make checkpoint persistence explicit
if RUN_MOUNT_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount skipped or unavailable:', exc)
else:
    print('Skipping Drive mount.')

if REQUIRE_DRIVE_FOR_CHECKPOINTS and not Path('/content/drive/MyDrive').exists():
    raise RuntimeError(
        'Google Drive is not mounted, so checkpoints would not be persistent. '
        'Set RUN_MOUNT_DRIVE=True and authorize Drive before training.'
    )

print('checkpoint/output root:', OUTPUT_ROOT)
print('backup root:', BACKUP_DIR)


In [ ]:
# Clone or update team repo
if RUN_CLONE_OR_UPDATE_REPO:
    if not REPO_DIR.exists():
        run(['git', 'clone', TEAM_REPO_URL, REPO_DIR])
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    checkout_cmd = ['git', 'checkout', TEAM_REPO_BRANCH]
    try:
        run(checkout_cmd, cwd=REPO_DIR)
    except subprocess.CalledProcessError:
        run(['git', 'checkout', '-B', TEAM_REPO_BRANCH, f'origin/{TEAM_REPO_BRANCH}'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', TEAM_REPO_BRANCH], cwd=REPO_DIR)
else:
    print('Skipping repo clone/update.')

if not RFDETR_DIR.exists():
    raise FileNotFoundError(f'RF_DETR_split_ver not found: {RFDETR_DIR}')
os.chdir(RFDETR_DIR)
print('cwd:', Path.cwd())


In [ ]:
# Install dependencies
if RUN_INSTALL_DEPS:
    # Colab already provides CUDA PyTorch. requirements.txt should usually be satisfied without replacing it.
    run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_DIR / 'requirements.txt'])
    run([sys.executable, '-m', 'pip', 'install', '-q', 'pycocotools', 'faster-coco-eval', 'opencv-python-headless', 'tqdm', 'gdown'])
else:
    print('Skipping dependency install.')


In [ ]:
# Runtime check
import torch

try:
    import rfdetr
    rfdetr_status = 'ok'
except Exception as exc:
    rfdetr_status = f'IMPORT FAILED: {exc}'

runtime_summary = {
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'rfdetr': rfdetr_status,
}
print(json.dumps(runtime_summary, indent=2, ensure_ascii=False))
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. Change Colab runtime type to GPU before training.')


In [ ]:
# Locate, download if needed, and extract dataset archive
ARCHIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
archive_path = DATASET_ARCHIVE_DIR / DATASET_ARCHIVE_NAME
archive_source = 'configured mounted Drive path'

if not archive_path.exists():
    print('Expected mounted archive not found:', archive_path)
    search_root = Path('/content/drive')
    candidates = list(search_root.rglob(DATASET_ARCHIVE_NAME))[:20] if search_root.exists() else []
    if candidates:
        archive_path = candidates[0]
        archive_source = 'found by scanning /content/drive'
        print('Using found archive:', archive_path)
    else:
        print('No readable mounted Drive copy found. Downloading from Drive file ID with gdown.')
        print('Dataset folder URL:', DATASET_ARCHIVE_FOLDER_URL)
        print('Dataset file URL:', DATASET_ARCHIVE_FILE_URL)
        try:
            import gdown
        except Exception:
            run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
            import gdown
        archive_path = ARCHIVE_CACHE_DIR / DATASET_ARCHIVE_NAME
        if archive_path.exists() and archive_path.stat().st_size > 0 and not FORCE_REEXTRACT_DATASET:
            print('Using cached downloaded archive:', archive_path)
        else:
            gdown.download(id=DATASET_ARCHIVE_FILE_ID, output=str(archive_path), quiet=False)
        archive_source = 'gdown file id download'

if not archive_path.exists() or archive_path.stat().st_size == 0:
    raise FileNotFoundError(f'Dataset archive not available after search/download: {archive_path}')

print('archive source:', archive_source)
print('archive:', archive_path)
print('archive size GB:', round(archive_path.stat().st_size / 1024**3, 3))

if RUN_EXTRACT_DATASET:
    already_ready = (DATASET_DIR / 'train' / '_annotations.coco.json').exists() and (DATASET_DIR / 'valid' / '_annotations.coco.json').exists()
    if already_ready and not FORCE_REEXTRACT_DATASET:
        print('Dataset already extracted:', DATASET_DIR)
    else:
        if DATASET_DIR.exists():
            shutil.rmtree(DATASET_DIR)
        DATASET_DIR.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archive_path, 'r:gz') as tar:
            tar.extractall(DATASET_DIR)
        print('Extracted to:', DATASET_DIR)
else:
    print('Skipping extraction. Using:', DATASET_DIR)


In [ ]:
# Dataset sanity checks
train_json = DATASET_DIR / 'train' / '_annotations.coco.json'
valid_json = DATASET_DIR / 'valid' / '_annotations.coco.json'
if not train_json.exists() or not valid_json.exists():
    raise FileNotFoundError(f'Missing train/valid annotations under {DATASET_DIR}')

summary_path = DATASET_DIR / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('dataset summary keys:', list(summary.keys()))
    if 'splits' in summary:
        display(pd.DataFrame(summary['splits']))
    if 'validation' in summary:
        print(json.dumps(summary['validation'].get('train_valid', summary['validation']), indent=2, ensure_ascii=False))
else:
    print('No summary.json found; continuing with COCO JSON checks.')

def load_coco(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

splits = {'train': load_coco(train_json), 'valid': load_coco(valid_json)}
rows = []
for split, data in splits.items():
    category_ids = sorted(int(c['id']) for c in data['categories'])
    ann_category_ids = sorted({int(a['category_id']) for a in data['annotations']})
    missing_files = []
    for image in data['images'][:]:
        if not (DATASET_DIR / split / image['file_name']).exists():
            missing_files.append(image['file_name'])
    rows.append({
        'split': split,
        'images': len(data['images']),
        'annotations': len(data['annotations']),
        'categories': len(category_ids),
        'ann_categories': len(ann_category_ids),
        'min_category_id': min(category_ids),
        'max_category_id': max(category_ids),
        'missing_files': len(missing_files),
    })
    if missing_files:
        raise FileNotFoundError(f'{split} has missing files, sample={missing_files[:5]}')

display(pd.DataFrame(rows))
category_counts = Counter(a['category_id'] for data in splits.values() for a in data['annotations'])
print('class annotation range:', min(category_counts.values()), 'to', max(category_counts.values()))
print('classes:', len(category_counts))


In [ ]:
# Create Colab CUDA config from the repo config
import yaml

base_config_path = RFDETR_DIR / BASE_CONFIG_NAME
if not base_config_path.exists():
    base_config_path = RFDETR_DIR / 'config_74_hidden45.yaml'
print('base config:', base_config_path)

config = yaml.safe_load(base_config_path.read_text(encoding='utf-8'))
config['data']['dataset_dir'] = str(DATASET_DIR)
# Source build folders are not needed when the extracted train/valid dataset already exists.
config['data']['base56_dir'] = ''
config['data']['hidden18_dir'] = ''
config['data']['test_images_dir'] = ''

config['model']['variant'] = MODEL_VARIANT
config['model']['tag'] = MODEL_TAG

train_cfg = config['train']
train_cfg.update({
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'grad_accum_steps': GRAD_ACCUM_STEPS,
    'seed': 42,
    'device': 'cuda',
    'pin_memory': PIN_MEMORY,
    'num_workers': NUM_WORKERS,
    'persistent_workers': bool(NUM_WORKERS > 0),
    'prefetch_factor': 2 if NUM_WORKERS > 0 else None,
    'checkpoint_interval': 1,
    'auto_resume': True,
    'resume': None,
    'eval_interval': 1,
    'eval_max_dets': 100,
    'compute_val_loss': True,
    'compute_test_loss': False,
    'run_test': False,
    'progress_bar': 'tqdm',
    'tensorboard': True,
    'multi_scale': ENABLE_MULTI_SCALE,
    'expanded_scales': False,
    'amp_dtype': 'auto',
})

TRAIN_AUG_CONFIG = {
    'RandomBrightnessContrast': {
        'brightness_limit': float(AUG_BRIGHTNESS_LIMIT),
        'contrast_limit': float(AUG_CONTRAST_LIMIT),
        'p': float(AUG_COLOR_P),
    },
    'Affine': {
        'scale': [float(AUG_SCALE_RANGE[0]), float(AUG_SCALE_RANGE[1])],
        'keep_ratio': True,
        'translate_percent': [float(AUG_TRANSLATE_PERCENT[0]), float(AUG_TRANSLATE_PERCENT[1])],
        'rotate': [-float(AUG_ROTATE_LIMIT_DEGREES), float(AUG_ROTATE_LIMIT_DEGREES)],
        'shear': 0,
        'border_mode': 4,
        'p': float(AUG_AFFINE_P),
    },
}
if ENABLE_TRAIN_AUGMENTATION:
    train_cfg['aug_config'] = TRAIN_AUG_CONFIG
    train_cfg['augmentation_backend'] = 'cpu'
else:
    train_cfg.pop('aug_config', None)
    train_cfg['augmentation_backend'] = 'cpu'

print('model tag:', MODEL_TAG)
print('train augmentation enabled:', ENABLE_TRAIN_AUGMENTATION)
if ENABLE_TRAIN_AUGMENTATION:
    print('train augmentation config:')
    print(yaml.safe_dump(TRAIN_AUG_CONFIG, allow_unicode=True, sort_keys=False))

if REQUIRE_DRIVE_FOR_CHECKPOINTS and not Path('/content/drive/MyDrive').exists():
    raise RuntimeError('Drive is not mounted; refusing to create checkpoint/output dirs outside persistent Drive.')

config['output']['local_output_dir'] = str(OUTPUT_ROOT)
config['output']['backup_dir'] = str(BACKUP_DIR)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print('checkpoints will be saved under:', OUTPUT_ROOT / MODEL_TAG)
print('best checkpoint backups will be saved under:', BACKUP_DIR)

config_path = RFDETR_DIR / CONFIG_NAME
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('wrote config:', config_path)
print(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))


In [ ]:
# Dry-run: validates config, dataset files, and train kwargs without training
if RUN_DRY_RUN:
    run([sys.executable, 'train_74_hidden45.py', '--config', config_path, '--dry-run'], cwd=RFDETR_DIR)
else:
    print('Skipping dry-run.')


In [ ]:
# Full train / resume / finalize
train_cmd = [sys.executable, 'train_74_hidden45.py', '--config', config_path]
if TRAIN_EPOCHS_OVERRIDE is not None:
    train_cmd += ['--epochs', str(TRAIN_EPOCHS_OVERRIDE)]

finalize_cmd = [sys.executable, 'train_74_hidden45.py', '--config', config_path, '--finalize-only']
print('train command:', ' '.join(map(str, train_cmd)))
print('finalize command:', ' '.join(map(str, finalize_cmd)))

if RUN_FINALIZE_ONLY:
    run(finalize_cmd, cwd=RFDETR_DIR)
elif RUN_FULL_TRAIN:
    env = os.environ.copy()
    env.setdefault('WANDB_DISABLED', 'true')
    env.setdefault('WANDB_MODE', 'disabled')
    run(train_cmd, cwd=RFDETR_DIR, env=env)
else:
    print('RUN_FULL_TRAIN=False. Set RUN_FULL_TRAIN=True in the parameter cell to launch or resume training.')


In [ ]:
# Metrics and checkpoints
import pandas as pd

output_dir = OUTPUT_ROOT / MODEL_TAG
backup_dir = BACKUP_DIR
metrics_csv = output_dir / 'metrics.csv'
print('output_dir:', output_dir)
print('backup_dir:', backup_dir)
print('metrics_csv:', metrics_csv)

if output_dir.exists():
    for path in sorted(output_dir.glob('*')):
        size = f'{path.stat().st_size / (1024 * 1024):.1f} MB' if path.is_file() else '<dir>'
        print(path.name, size)
else:
    print('No output_dir yet.')

if metrics_csv.exists():
    metrics_df = pd.read_csv(metrics_csv)
    cols = [c for c in ['epoch', 'val/mAP_75', 'val/mAP_50_95', 'val/mAP_50', 'val/mAR', 'val/loss', 'train/loss'] if c in metrics_df.columns]
    per_epoch = metrics_df.groupby('epoch').last().reset_index()[cols]
    display(per_epoch.tail(20))
    if 'val/mAP_75' in per_epoch.columns and per_epoch['val/mAP_75'].notna().any():
        best_idx = per_epoch['val/mAP_75'].idxmax()
        best = per_epoch.loc[best_idx]
        print(f"Best mAP@0.75: epoch={int(best['epoch'])}, score={float(best['val/mAP_75']):.6f}")
        print('best mAP75 checkpoint:', output_dir / 'checkpoint_best_map75.ckpt')
else:
    print('metrics.csv not found yet. Run training first.')

map75_summary_path = output_dir / 'map75_summary.json'
if map75_summary_path.exists():
    print('map75 summary:')
    print(map75_summary_path.read_text(encoding='utf-8'))



## Test Inference & Submission

After training finishes, set `RUN_TEST_INFERENCE = True` and rerun these cells. The notebook loads the best `mAP@0.75` checkpoint, predicts the original competition test images, and writes a Kaggle-style `submission.csv` to the Drive output folder.


In [ ]:

# Locate readable original test images
from pathlib import Path

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}


def numeric_image_id_from_path(path: Path) -> int:
    stem = path.stem
    if stem.isdigit():
        return int(stem)
    digits = ''.join(ch for ch in stem if ch.isdigit())
    if digits:
        return int(digits)
    raise ValueError(f'Cannot parse numeric image_id from filename: {path.name}')


def list_image_paths(folder: Path):
    if folder is None or not folder.exists():
        return []
    return sorted(
        [p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS and p.is_file()],
        key=numeric_image_id_from_path,
    )


def readable_image_paths(folder: Path):
    paths = list_image_paths(folder)
    if not paths:
        return []
    try:
        from PIL import Image
        with Image.open(paths[0]) as im:
            im.verify()
    except Exception as exc:
        print(f'Image folder is present but not readable: {folder} ({exc})')
        return []
    return paths


def find_or_download_test_images():
    candidates = [
        DATASET_DIR / 'test',
        DRIVE_PROJECT_DIR / 'sprint_ai_project1_data' / 'test_images',
        DRIVE_PROJECT_DIR / 'test_images',
        TEST_DATA_DIR / 'test_images',
        TEST_DATA_DIR,
    ]
    for candidate in candidates:
        paths = readable_image_paths(candidate)
        if paths:
            print('test image source:', candidate)
            return candidate, paths

    drive_root = Path('/content/drive')
    if drive_root.exists():
        checked = 0
        for candidate in drive_root.rglob('test_images'):
            checked += 1
            paths = readable_image_paths(candidate)
            if paths:
                print('test image source found by Drive scan:', candidate)
                return candidate, paths
            if checked >= 50:
                break

    print('Readable mounted test_images folder not found. Downloading from Drive folder with gdown.')
    try:
        import gdown
    except Exception:
        run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
        import gdown

    TEST_DATA_DIR.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(url=TEST_IMAGES_FOLDER_URL, output=str(TEST_DATA_DIR), quiet=False, use_cookies=False)

    search_candidates = [TEST_DATA_DIR, TEST_DATA_DIR / 'test_images'] + [p for p in TEST_DATA_DIR.rglob('test_images')]
    for candidate in search_candidates:
        paths = readable_image_paths(candidate)
        if paths:
            print('test image source downloaded:', candidate)
            return candidate, paths

    raise FileNotFoundError('Could not locate readable test images after Drive search/download.')


if RUN_TEST_INFERENCE:
    TEST_IMG_DIR, test_paths = find_or_download_test_images()
    print('num test images:', len(test_paths))
    print('sample:', [p.name for p in test_paths[:5]])
else:
    TEST_IMG_DIR = None
    test_paths = []
    print('Skipping test image lookup. Set RUN_TEST_INFERENCE=True to create submission.csv.')


In [ ]:

# Run RF-DETR test inference and write submission.csv
if RUN_TEST_INFERENCE:
    import re
    import sys
    import numpy as np
    from PIL import Image
    from tqdm.auto import tqdm

    if str(RFDETR_DIR) not in sys.path:
        sys.path.insert(0, str(RFDETR_DIR))
    from model import get_rfdetr_model

    output_dir = OUTPUT_ROOT / MODEL_TAG
    backup_dir = BACKUP_DIR
    SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

    def latest_epoch_checkpoint(folder: Path):
        pattern = re.compile(r'checkpoint_(\d+)\.ckpt$')
        candidates = []
        if not folder.exists():
            return None
        for path in folder.glob('checkpoint_*.ckpt'):
            match = pattern.match(path.name)
            if match:
                candidates.append((int(match.group(1)), path.stat().st_mtime, path))
        return max(candidates, key=lambda item: (item[0], item[1]))[2] if candidates else None

    def resolve_inference_checkpoint():
        candidates = [
            output_dir / 'checkpoint_best_map75.ckpt',
            backup_dir / f'{MODEL_TAG}_best_map75.ckpt',
            output_dir / 'checkpoint_best_total.pth',
            backup_dir / f'{MODEL_TAG}_best.pth',
        ]
        latest = latest_epoch_checkpoint(output_dir)
        if latest is not None:
            candidates.append(latest)
        for candidate in candidates:
            if candidate.exists() and candidate.stat().st_size > 0:
                return candidate
        raise FileNotFoundError(
            'No checkpoint found for submission. Finish training first, or run finalize-only to create checkpoint_best_map75.ckpt.'
        )

    checkpoint_path = resolve_inference_checkpoint()
    print('submission checkpoint:', checkpoint_path)

    mapping_path = DATASET_DIR / 'category_mapping.csv'
    if not mapping_path.exists():
        raise FileNotFoundError(f'Missing category mapping: {mapping_path}')
    mapping_df = pd.read_csv(mapping_path, encoding='utf-8-sig')
    internal_to_category = {
        int(row.rfdetr_internal_label): int(row.category_id)
        for row in mapping_df.itertuples(index=False)
    }
    valid_category_ids = set(internal_to_category.values())

    def to_submission_category_id(raw_label):
        label = int(raw_label)
        if label in valid_category_ids:
            return label
        if label in internal_to_category:
            return internal_to_category[label]
        if (label - 1) in internal_to_category:
            return internal_to_category[label - 1]
        raise KeyError(f'Cannot map model class_id={label} to original category_id')

    model = get_rfdetr_model(MODEL_VARIANT, checkpoint_path=str(checkpoint_path))

    def predict_test_image(img_path: Path, score_thr=TEST_SCORE_THR, max_det=TEST_MAX_DET_PER_IMAGE):
        with Image.open(img_path) as im:
            image = im.convert('RGB')
            img_w, img_h = image.size
            detections = model.predict(image, threshold=score_thr)

        boxes = np.asarray(getattr(detections, 'xyxy', []), dtype=float)
        if boxes.size == 0:
            return []
        boxes = boxes.reshape(-1, 4)
        scores = np.asarray(getattr(detections, 'confidence', np.ones(len(boxes))), dtype=float).reshape(-1)
        labels = np.asarray(getattr(detections, 'class_id', np.zeros(len(boxes))), dtype=int).reshape(-1)

        order = np.argsort(-scores)
        if max_det is not None:
            order = order[:max_det]

        preds = []
        for idx in order:
            x1, y1, x2, y2 = boxes[idx].tolist()
            x1 = max(0.0, min(float(x1), float(img_w - 1)))
            y1 = max(0.0, min(float(y1), float(img_h - 1)))
            x2 = max(0.0, min(float(x2), float(img_w)))
            y2 = max(0.0, min(float(y2), float(img_h)))
            bw = x2 - x1
            bh = y2 - y1
            if bw <= 0 or bh <= 0:
                continue
            preds.append({
                'category_id': to_submission_category_id(labels[idx]),
                'bbox_x': x1,
                'bbox_y': y1,
                'bbox_w': bw,
                'bbox_h': bh,
                'score': float(scores[idx]),
            })
        return preds

    submission_rows = []
    annotation_id = 1
    for img_path in tqdm(test_paths, desc='test inference'):
        image_id = numeric_image_id_from_path(img_path)
        preds = predict_test_image(img_path)
        for pred in preds:
            submission_rows.append({
                'annotation_id': int(annotation_id),
                'image_id': int(image_id),
                'category_id': int(pred['category_id']),
                'bbox_x': round(float(pred['bbox_x']), 2),
                'bbox_y': round(float(pred['bbox_y']), 2),
                'bbox_w': round(float(pred['bbox_w']), 2),
                'bbox_h': round(float(pred['bbox_h']), 2),
                'score': round(float(pred['score']), 6),
            })
            annotation_id += 1

    submission = pd.DataFrame(
        submission_rows,
        columns=['annotation_id', 'image_id', 'category_id', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score'],
    )
    SUBMISSION_PATH = SUBMISSION_DIR / SUBMISSION_FILE_NAME
    submission.to_csv(SUBMISSION_PATH, index=False)

    print('saved:', SUBMISSION_PATH)
    print('shape:', submission.shape)
    display(submission.head(20))
else:
    print('Skipping test inference. Set RUN_TEST_INFERENCE=True after training/finalize.')


In [ ]:

# Submission sanity checks
if RUN_TEST_INFERENCE:
    required_columns = ['annotation_id', 'image_id', 'category_id', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score']
    assert list(submission.columns) == required_columns
    assert len(submission) > 0, 'submission is empty; lower TEST_SCORE_THR or inspect checkpoint/test images'
    assert submission['annotation_id'].is_unique
    assert submission['bbox_w'].min() > 0
    assert submission['bbox_h'].min() > 0
    assert submission['score'].between(0, 1).all()
    assert set(submission['category_id']).issubset(valid_category_ids)

    expected_image_ids = {numeric_image_id_from_path(p) for p in test_paths}
    predicted_image_ids = set(submission['image_id'].astype(int))
    missing_prediction_images = sorted(expected_image_ids - predicted_image_ids)
    print('submission rows:', len(submission))
    print('test images:', len(expected_image_ids))
    print('images with at least one prediction:', len(predicted_image_ids))
    print('images without predictions:', len(missing_prediction_images))
    if missing_prediction_images:
        print('missing prediction sample:', missing_prediction_images[:20])
    display(submission.groupby('image_id').size().describe())
    print('submission format looks good')
else:
    print('Skipping submission checks.')


In [ ]:
# Optional TensorBoard
if RUN_SHOW_TENSORBOARD:
    ip = get_ipython()
    ip.run_line_magic('load_ext', 'tensorboard')
    ip.run_line_magic('tensorboard', f'--logdir {OUTPUT_ROOT}')
else:
    print('Skipping TensorBoard. Set RUN_SHOW_TENSORBOARD=True to show it.')


## Augmentation Notes

This notebook now defaults to an augmented run so the baseline checkpoint folder is not overwritten:

```python
ENABLE_TRAIN_AUGMENTATION = True
AUGMENTATION_NAME = 'aug_scale150_rot90_v1'
MODEL_TAG = f'{BASE_MODEL_TAG}_{AUGMENTATION_NAME}'
```

Current train-only augmentation:

- scale `[0.85, 1.50]`
- translate `[-0.04, 0.04]`
- rotate `[-90, 90]` degrees via `AUG_ROTATE_LIMIT_DEGREES`
- brightness/contrast `0.08`, probability `0.25`
- no flip, because mirrored pill imprints are unrealistic

Validation and test images stay unchanged. To return to the baseline run, set:

```python
ENABLE_TRAIN_AUGMENTATION = False
```

Checkpoint safety remains on: `checkpoint_interval=1`, `auto_resume=True`, and all outputs are written under Drive `OUTPUT_ROOT / MODEL_TAG`; best mAP@0.75 copies go to `BACKUP_DIR` with the model tag in the filename.


## Expected Outputs

After a completed full run, outputs are isolated by the current `MODEL_TAG`:

- `metrics.csv` under `OUTPUT_ROOT / MODEL_TAG`
- RF-DETR regular best checkpoint: `checkpoint_best_total.pth`
- wrapper-selected best mAP@0.75 checkpoint: `checkpoint_best_map75.ckpt`
- Drive backup copies under `BACKUP_DIR`
- `map75_summary.json` with the selected epoch and score
- when `RUN_TEST_INFERENCE=True`, submission CSV under `SUBMISSION_DIR`
- augmentation settings are printed into the generated YAML config cell output

If Colab disconnects, rerun the notebook. With `auto_resume: true`, training resumes from the latest `checkpoint_N.ckpt` in the Drive output folder.
